## Testing the building the model codes of bigram.py and others :
----

In [1]:
import torch 
import torch.nn as nn
from torch.nn import functional as F

# ## Hyperparameters 
# batch_size = 32
# block_size = 8
# max_iters = 3000
# eval_interval = 300
# learning_rate = 1e-2
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# eval_iters = 200

# ##-------------------

# torch.manual_seed(1337)


# ## reading the dataset :

# with open("/home/aditya/coding_files/python/env_conda/experiments/nano_gpt/input_tiny_shakespear.txt","r",encoding="utf-8") as f: 
#     text = f.read()


# ## No of unique chars
# #  here are all the unique characters that occur in this text
# chars = sorted(list(set(text)))
# vocab_size = len(chars)
# # create a mapping from characters to integers
# stoi = { ch:i for i,ch in enumerate(chars) }
# itos = { i:ch for i,ch in enumerate(chars) }
# encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
# decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# # Train and test splits
# data = torch.tensor(encode(text), dtype=torch.long)
# n = int(0.9*len(data)) # first 90% will be train, rest val
# train_data = data[:n]
# val_data = data[n:]

# # data loading
# def get_batch(split):
#     # generate a small batch of data of inputs x and targets y
#     data = train_data if split == 'train' else val_data
#     ix = torch.randint(len(data) - block_size, (batch_size,))
#     x = torch.stack([data[i:i+block_size] for i in ix])
#     y = torch.stack([data[i+1:i+block_size+1] for i in ix])
#     x, y = x.to(device), y.to(device)
#     return x, y
# # ----------------------------
# ## torch no grad function 
# @torch.no_grad()
# def estimate_loss():
#     out = {}
#     model.eval()
#     for split in ['train', 'val']:
#         losses = torch.zeros(eval_iters)
#         for k in range(eval_iters):
#             X, Y = get_batch(split)
#             logits, loss = model(X, Y)
#             losses[k] = loss.item()
#         out[split] = losses.mean()
#     model.train()
#     return out

# # super simple bigram model
# class BigramLanguageModel(nn.Module):

#     def __init__(self, vocab_size):
#         super().__init__()
#         # each token directly reads off the logits for the next token from a lookup table
#         self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

#     def forward(self, idx, targets=None):

#         # idx and targets are both (B,T) tensor of integers
#         logits = self.token_embedding_table(idx) # (B,T,C)

#         if targets is None:
#             loss = None
#         else:
#             B, T, C = logits.shape
#             logits = logits.view(B*T, C)
#             targets = targets.view(B*T)
#             loss = F.cross_entropy(logits, targets)

#         return logits, loss

#     def generate(self, idx, max_new_tokens):
#         # idx is (B, T) array of indices in the current context
#         for _ in range(max_new_tokens):
#             # get the predictions
#             logits, loss = self(idx)
#             # focus only on the last time step
#             logits = logits[:, -1, :] # becomes (B, C)
#             # apply softmax to get probabilities
#             probs = F.softmax(logits, dim=-1) # (B, C)
#             # sample from the distribution
#             idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
#             # append sampled index to the running sequence
#             idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
#         return idx

# model = BigramLanguageModel(vocab_size)
# m = model.to(device)

# # create a PyTorch optimizer
# optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# for iter in range(max_iters):

#     # every once in a while evaluate the loss on train and val sets
#     if iter % eval_interval == 0:
#         losses = estimate_loss()
#         print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

#     # sample a batch of data
#     xb, yb = get_batch('train')

#     # evaluate the loss
#     logits, loss = model(xb, yb)
#     optimizer.zero_grad(set_to_none=True)
#     loss.backward()
#     optimizer.step()

# # generate from the model
# context = torch.zeros((1, 1), dtype=torch.long, device=device)
# print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


## The mathematical trick for self attention:
- matrix multiplication makes the average of previous ad current element easier 
- instead of the traditional for loop we use the matrix mul
----

In [3]:
### consider the following toy operation 
torch.manual_seed(1337)
B, T, C = 4, 8 ,2  ## Batch, Time, Channels 
x = torch.randn(B, T, C)
x.shape


torch.Size([4, 8, 2])

In [13]:
## we want x[b,t] = mean_{i<=t} x [b,i]
xbow = torch.zeros((B, T, C)) ## X-bag of words 
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] ## [t,c]
        xbow[b,t] = torch.mean(xprev, 0)




### matrix multiplication trick :
---

In [ ]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3,3)) ## creates thelower triangular matrix of one(1) of dim:3 x 3

a= a/torch.sum(a,1,keepdim= True) ## mean / avg of matrix a with col no 

b = torch.randint(0,10,(3,2)).float()  ## creates the random matrix of dim (3x2) for random value betwn (1->10)

c = a @ b ##  matrix multiplication

print("a :")
print(a)
print('=='*30)
print("b :")
print(b)
print('=='*30)
print("c :")
print(c)
print('=='*30)


a :
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b :
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c :
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
wei= torch.tril(torch.ones(T, T)) ## wei = weights
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x ## (B,T,T) @ (B,T,C)
torch.allclose(xbow,xbow2)

False

In [22]:
xbow[0],xbow2[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [ ]:
## v3 for self attention use softmax 

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei =  wei.masked_fill(tril == 0,float('-inf'))
wei = F.softmax(wei,dim= -1)
xbow3 = wei @ x
# torch.allclose(xbow, xbow3)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])